# Notebook 1: Exploratory Data Analysis for Customer Churn

This notebook is designed as a teaching notebook. Instead of jumping straight into modeling, we use it to understand the business problem, inspect the raw data, surface modeling risks, and build evidence for the feature engineering choices used in the production churn system.

The guiding question is simple: **what early warning signs tell us that a customer is becoming more likely to leave?**


## Learning Outcomes

By the end of this notebook, a learner should be able to:

- explain the business importance of churn prediction in plain language
- audit a real customer dataset for data quality issues before modeling
- identify which columns are useful, which are risky, and which should never be used for training
- reason about missing values, dates, times, behavior variables, and categorical signals
- translate exploratory findings into concrete feature engineering and modeling hypotheses
- understand why the deployed system uses a final churn scale from **1** (lowest risk) to **5** (highest risk)


## Notebook Roadmap

We will move through the same thought process a production data scientist should use:

1. Load the training and test data.
2. Validate the dataset structure.
3. Inspect the target and confirm the business-facing risk scale.
4. Quantify missing data and leakage risks.
5. Explore the most important behavioral, lifecycle, and service-quality signals.
6. Summarize what the EDA tells us about how churn should be modeled.


In [ ]:
from pathlib import Path
import json
import os
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "config.json").exists():
            return candidate
    raise FileNotFoundError("Could not locate the churn project root.")

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root resolved to: {PROJECT_ROOT}")


In [ ]:
from app.core.config import get_settings
from src.data.load_data import load_test_data, load_train_data
from src.data.target_normalization import (
    describe_target_normalization,
    normalize_target_frame,
    resolve_target_normalization_map,
)
from src.data.validate_data import validate_inference_frame, validate_training_frame

settings = get_settings()
target_column = settings.target_column
normalization_map = resolve_target_normalization_map(settings.target_normalization_map)


In [ ]:
train_df = load_train_data()
test_df = load_test_data()
train_normalized = normalize_target_frame(train_df, target_column, normalization_map)

print(f"Training rows, columns: {train_df.shape}")
print(f"Test rows, columns: {test_df.shape}")
display(train_df.head())


## 1. Structural Validation Comes Before Insight

A common beginner mistake is to start plotting immediately. In real work, we first check whether the dataset is structurally usable. If required columns are missing, IDs are duplicated, or the target is incomplete, any later analysis becomes unreliable.

In production, this same validation step protects the API and batch-scoring pipeline from malformed input.


In [ ]:
training_report = validate_training_frame(train_df, target_column)
test_report = validate_inference_frame(test_df)

validation_summary = pd.DataFrame(
    [
        {"dataset": "training", **training_report.to_dict()},
        {"dataset": "test", **test_report.to_dict()},
    ]
)
display(validation_summary)


In [ ]:
schema_audit = pd.DataFrame(
    {
        "column": train_df.columns,
        "dtype": [str(train_df[column].dtype) for column in train_df.columns],
        "non_null": [int(train_df[column].notna().sum()) for column in train_df.columns],
        "missing": [int(train_df[column].isna().sum()) for column in train_df.columns],
        "missing_pct": [round(train_df[column].isna().mean() * 100, 2) for column in train_df.columns],
        "unique_values": [int(train_df[column].nunique(dropna=True)) for column in train_df.columns],
        "example_value": [train_df[column].dropna().iloc[0] if train_df[column].dropna().shape[0] else None for column in train_df.columns],
    }
).sort_values(["missing_pct", "unique_values"], ascending=[False, False])

display(schema_audit)


## 2. Understanding the Target

The raw dataset uses a legacy encoding for low churn risk, but the production system standardizes the target to a business-friendly scale from **1** to **5**:

- **1** = lowest churn risk
- **2** = low-to-moderate churn risk
- **3** = moderate churn risk
- **4** = elevated churn risk
- **5** = highest churn risk

This matters because every downstream artifact, API response, dashboard, and recommendation should speak the same language to business users.


In [ ]:
normalization_summary = describe_target_normalization(
    train_df[target_column],
    train_normalized[target_column],
    normalization_map,
)

target_distribution = (
    train_normalized[target_column]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("risk_class")
    .reset_index(name="customers")
)
target_distribution["share_pct"] = (target_distribution["customers"] / len(train_normalized) * 100).round(2)

display(pd.DataFrame([normalization_summary]))
display(target_distribution)

fig = px.bar(
    target_distribution,
    x="risk_class",
    y="customers",
    text="share_pct",
    title="Final 1-to-5 Churn Risk Distribution",
    color="risk_class",
    color_discrete_sequence=px.colors.sequential.YlOrBr,
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(showlegend=False, xaxis_title="Normalized churn class", yaxis_title="Customers")
fig.show()


### Teaching Note

Before choosing a model family, always inspect:

- how many target classes exist
- whether the classes are ordered
- whether the class distribution is balanced enough to trust accuracy alone

In this project, the target is clearly ordered and low-cardinality, which is why the production system treats it as an **ordinal multiclass** problem instead of a regression problem.


## 3. Missing Data Analysis

Missingness is never just a cleaning problem. It can also be a behavioral signal.

For example:

- missing `points_in_wallet` may suggest weak loyalty-program participation
- missing `region_category` may reflect incomplete customer profiling
- missing `preferred_offer_types` may reflect low marketing engagement or sparse records

Good EDA asks two questions at once:

1. How much data is missing?
2. Could the missingness itself carry information?


In [ ]:
missing_summary = (
    train_df.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda frame: (frame["missing_count"] / len(train_df) * 100).round(2))
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
    .reset_index(names="column")
)
display(missing_summary)

fig = px.bar(
    missing_summary,
    x="missing_count",
    y="column",
    orientation="h",
    text="missing_pct",
    title="Columns with Missing Values in the Training Set",
    color="missing_pct",
    color_continuous_scale="YlOrBr",
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(xaxis_title="Missing records", yaxis_title="Column")
fig.show()


## 4. Leakage and Non-Generalizable Columns

Not every column with predictive power should be used. A mature model avoids fields that are unstable, personally identifying, or too specific to memorize.

In this project, the production system excludes:

- `customer_id`
- `Name`
- `security_no`
- `referral_id`

These are retained only for display, record tracking, or operational context, not for learning patterns.


In [ ]:
leakage_review = pd.DataFrame(
    {
        "column": settings.leakage_columns,
        "unique_values": [train_df[column].nunique(dropna=True) for column in settings.leakage_columns],
        "unique_ratio": [round(train_df[column].nunique(dropna=True) / len(train_df), 3) for column in settings.leakage_columns],
        "recommended_action": [
            "Drop from modeling, retain only as business identifier" for _ in settings.leakage_columns
        ],
    }
)
display(leakage_review)


## 5. Parsing Time and Lifecycle Signals

Churn is often a lifecycle problem. Customers who joined recently behave differently from long-tenure customers, and the timing of their visits can reveal habit strength.

We inspect the raw date and time fields here so the later feature engineering steps feel motivated rather than arbitrary.


In [ ]:
eda_dates = train_normalized.copy()
eda_dates["joining_date_parsed"] = pd.to_datetime(eda_dates["joining_date"], errors="coerce")
eda_dates["last_visit_time_parsed"] = pd.to_datetime(
    eda_dates["last_visit_time"],
    format="%H:%M:%S",
    errors="coerce",
)
eda_dates["joining_year"] = eda_dates["joining_date_parsed"].dt.year
eda_dates["joining_month"] = eda_dates["joining_date_parsed"].dt.month
eda_dates["joining_weekday"] = eda_dates["joining_date_parsed"].dt.day_name()
eda_dates["visit_hour"] = eda_dates["last_visit_time_parsed"].dt.hour

temporal_quality = pd.DataFrame(
    {
        "field": ["joining_date", "last_visit_time"],
        "successfully_parsed": [
            int(eda_dates["joining_date_parsed"].notna().sum()),
            int(eda_dates["last_visit_time_parsed"].notna().sum()),
        ],
        "parse_rate_pct": [
            round(eda_dates["joining_date_parsed"].notna().mean() * 100, 2),
            round(eda_dates["last_visit_time_parsed"].notna().mean() * 100, 2),
        ],
    }
)
display(temporal_quality)
display(eda_dates[["joining_date", "joining_date_parsed", "last_visit_time", "last_visit_time_parsed"]].head())


## 6. Core Numeric Behavior Signals

Several fields describe how active and valuable a customer appears:

- `days_since_last_login`
- `avg_time_spent`
- `avg_transaction_value`
- `avg_frequency_login_days`
- `points_in_wallet`
- `age`

A useful EDA habit is to look at both overall distributions and class-level averages. The overall distribution tells us whether a variable is noisy or skewed, while the class view tells us whether it actually differentiates risk.


In [ ]:
numeric_columns = [
    "age",
    "days_since_last_login",
    "avg_time_spent",
    "avg_transaction_value",
    "avg_frequency_login_days",
    "points_in_wallet",
]

numeric_summary = train_normalized[numeric_columns].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
display(numeric_summary)


In [ ]:
class_profile = (
    train_normalized.groupby(target_column)[numeric_columns]
    .mean()
    .round(2)
)
standardized_profile = (class_profile - class_profile.mean()) / class_profile.std(ddof=0)
display(class_profile)

fig = px.imshow(
    standardized_profile.T,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdYlGn_r",
    title="How Numeric Signals Shift Across Churn Classes (standardized)",
    labels={"x": "Churn class", "y": "Numeric feature", "color": "Relative level"},
)
fig.show()


### Interpreting the Numeric Heatmap

In the heatmap above:

- warmer values suggest the feature is relatively **higher** for that class
- cooler values suggest the feature is relatively **lower** for that class

This is a fast way to identify whether higher-risk customers are less engaged, less valuable, or less loyal.


## 7. Categorical Signals: Customer Type, Channel, and Offer Behavior

Categorical columns often tell the most actionable story because they map directly to operational segments: membership tier, feedback type, internet option, offer preference, and complaint status are all categories that a business team can act on.


In [ ]:
def churn_mix(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    counts = frame.groupby([column, target_column]).size().reset_index(name="customers")
    totals = counts.groupby(column)["customers"].transform("sum")
    counts["share_within_segment"] = counts["customers"] / totals
    return counts.sort_values([column, target_column])

membership_mix = churn_mix(train_normalized, "membership_category")
display(membership_mix.head(20))

fig = px.bar(
    membership_mix,
    x="membership_category",
    y="share_within_segment",
    color=target_column,
    barmode="stack",
    title="Membership Category Mix by Churn Class",
    color_discrete_sequence=px.colors.sequential.YlOrBr,
)
fig.update_layout(yaxis_title="Share within membership segment", xaxis_title="Membership category")
fig.show()


In [ ]:
feedback_mix = churn_mix(train_normalized, "feedback")
top_feedback_levels = (
    train_normalized["feedback"].value_counts().head(8).index.tolist()
)
feedback_plot = feedback_mix[feedback_mix["feedback"].isin(top_feedback_levels)]

fig = px.bar(
    feedback_plot,
    x="feedback",
    y="share_within_segment",
    color=target_column,
    barmode="stack",
    title="Feedback Categories and Their Churn Mix",
    color_discrete_sequence=px.colors.sequential.YlOrBr,
)
fig.update_layout(yaxis_title="Share within feedback category", xaxis_title="Feedback")
fig.show()


## 8. Service Recovery Signals: Complaints and Experience Quality

Complaint and feedback variables are especially valuable because they connect directly to business action. If customers with unresolved complaints consistently appear in higher-risk classes, a company does not need more theory. It needs a better recovery process.


In [ ]:
complaint_summary = (
    train_normalized.groupby(["past_complaint", "complaint_status", target_column])
    .size()
    .reset_index(name="customers")
)
complaint_summary["segment_total"] = complaint_summary.groupby(["past_complaint", "complaint_status"])["customers"].transform("sum")
complaint_summary["share_within_segment"] = complaint_summary["customers"] / complaint_summary["segment_total"]
display(complaint_summary.head(30))

fig = px.bar(
    complaint_summary,
    x="complaint_status",
    y="share_within_segment",
    color=target_column,
    facet_col="past_complaint",
    barmode="stack",
    title="Complaint Resolution Status and Churn Mix",
    color_discrete_sequence=px.colors.sequential.YlOrBr,
)
fig.update_layout(yaxis_title="Share within complaint segment", xaxis_title="Complaint status")
fig.show()


## 9. Referral, Region, and Access Channel Patterns

Churn is also shaped by how customers enter and interact with the service. Referral users may behave differently from non-referral users, and channel differences can hint at friction in product access or marketing alignment.


In [ ]:
channel_columns = [
    "joined_through_referral",
    "region_category",
    "medium_of_operation",
    "internet_option",
    "preferred_offer_types",
]

channel_insights = []
for column in channel_columns:
    grouped = (
        train_normalized.groupby(column)[target_column]
        .agg(["count", "mean"])
        .reset_index()
        .sort_values("count", ascending=False)
    )
    grouped["column"] = column
    channel_insights.append(grouped)

channel_insights = pd.concat(channel_insights, ignore_index=True)
display(channel_insights.head(25))


## 10. Temporal Lifecycle Patterns

Dates are most useful when they become lifecycle questions:

- Do customers from certain joining periods look riskier?
- Are some visit hours associated with stronger or weaker churn risk?
- Does tenure appear protective, or do long-tenure customers also drift if engagement weakens?


In [ ]:
join_year_risk = (
    eda_dates.groupby("joining_year")[target_column]
    .mean()
    .reset_index(name="avg_churn_class")
    .dropna()
)
visit_hour_risk = (
    eda_dates.groupby("visit_hour")[target_column]
    .mean()
    .reset_index(name="avg_churn_class")
    .dropna()
)

fig_year = px.line(
    join_year_risk,
    x="joining_year",
    y="avg_churn_class",
    markers=True,
    title="Average Churn Class by Joining Year",
)
fig_year.update_layout(xaxis_title="Joining year", yaxis_title="Average churn class")
fig_year.show()

fig_hour = px.line(
    visit_hour_risk,
    x="visit_hour",
    y="avg_churn_class",
    markers=True,
    title="Average Churn Class by Visit Hour",
)
fig_hour.update_layout(xaxis_title="Visit hour", yaxis_title="Average churn class")
fig_hour.show()


## 11. A Simple High-Risk vs Low-Risk Profile Comparison

One of the most practical teaching exercises is to compare the safest customers with the riskiest customers. This helps learners move from "interesting variables" to "actionable differences".


In [ ]:
comparison_features = [
    "days_since_last_login",
    "avg_time_spent",
    "avg_transaction_value",
    "avg_frequency_login_days",
    "points_in_wallet",
]

low_risk_profile = train_normalized.loc[train_normalized[target_column] == 1, comparison_features].mean()
high_risk_profile = train_normalized.loc[train_normalized[target_column] == 5, comparison_features].mean()

class_comparison = pd.DataFrame(
    {
        "class_1_mean": low_risk_profile,
        "class_5_mean": high_risk_profile,
        "difference_class5_minus_class1": high_risk_profile - low_risk_profile,
    }
).round(2)
display(class_comparison)


## 12. EDA Conclusions

At this point, we can already justify several production design choices:

- **behavioral engagement matters**: login recency, time spent, and login frequency clearly deserve engineered features
- **service experience matters**: complaint and feedback variables carry operationally meaningful churn signal
- **loyalty signals matter**: wallet points and transaction value help distinguish high-value, lower-risk customers from weaker relationships
- **lifecycle matters**: tenure and visit timing are worth transforming into reusable features
- **not all columns are fair game**: identifiers and PII should be excluded even if they appear predictive

These insights flow directly into the feature engineering and modeling notebook that follows.


## What Comes Next

In the next notebook, we turn these exploratory findings into a production-safe feature pipeline, compare multiple candidate models, measure overfitting risk, persist the best artifact, and connect predictions to business-friendly retention recommendations.
